# Eight Weeks Without the REM

*July 5 – August 17, 2025: When the REM suspended service between Brossard and Gare Centrale, the RTL put a shuttle bus network in place for the sake of service continuity. During that downtime, where did the people that relied on the REM go? What did they do instead?*

*This notebook uses RTL GTFS data and Bixi trip records to assess the shuttle system on coverage, frequency, and travel time, then looks for a behavioural signal in Bixi ridership near REM stations during the closure.*

---
## 1. Setup

In [9]:
import zipfile
from pathlib import Path

import pandas as pd
import gtfs_kit as gk

In [10]:
# define data paths
DATA_DIR = Path("data")
GTFS_DIR = DATA_DIR / "gtfs"
BIXI_DIR = DATA_DIR / "bixi"
REM_DIR  = DATA_DIR / "rem"

# create directories for data
for d in [GTFS_DIR, BIXI_DIR, REM_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
# set up unzip function that takes a Path as input
def unzip(zip_path: Path) -> None:
    extract_dir = zip_path.with_suffix("")
    if extract_dir.exists():
        print(f"  {extract_dir.name}/ ...already extracted, skipping")
        return
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_dir)
    print(f"  {zip_path.name} → {extract_dir.name}/")

# unzip RTL GTFS
print("RTL GTFS feeds:")
for zp in sorted(GTFS_DIR.glob("*.zip")):
    unzip(zp)

# unzip Bixi
print("\nBixi trip data:")
for zp in sorted(BIXI_DIR.glob("*.zip")):
    unzip(zp)

# unzip REM GTFS
print("\nREM GTFS feeds:")
for zp in sorted(REM_DIR.glob("*.zip")):
    unzip(zp)

RTL GTFS feeds:
  20250623/ ...already extracted, skipping
  20250707/ ...already extracted, skipping
  20250825/ ...already extracted, skipping

Bixi trip data:
  DonneesOuvertes2024_010203040506070809101112/ ...already extracted, skipping
  DonneesOuvertes2025_010203040506070809101112/ ...already extracted, skipping

REM GTFS:
  gtfs.zip → gtfs/


In [13]:
# summarize data after being unzipped

print("RTL GTFS data overview:")
for d in sorted(GTFS_DIR.iterdir()):
    if d.is_dir():
        print(f"  {d.name}: {len(list(d.iterdir()))} files")

print("\nBixi data overview:")
for d in sorted(BIXI_DIR.iterdir()):
    if d.is_dir():
        csvs = list(d.glob("*.csv"))
        print(f"  {d.name}: {len(csvs)} CSV files")

print("\nREM GTFS data overview:")
for d in sorted(REM_DIR.iterdir()):
    if d.is_dir():
        print(f"  {d.name}: {len(list(d.iterdir()))} files")

RTL GTFS data overview:
  20250623: 12 files
  20250707: 12 files
  20250825: 12 files

Bixi data overview:
  DonneesOuvertes2024_010203040506070809101112: 1 CSV files
  DonneesOuvertes2025_010203040506070809101112: 1 CSV files

REM GTFS data overview:
  gtfs: 12 files


---
## 2. The REM Baseline

Before July 5, the REM's Brossard branch connected five stations from the South Shore to downtown in under 25 minutes. This section profiles that service — stop sequence, scheduled travel time, and peak headways — to establish what riders actually lost when service suspended.

*Note: this feed reflects current REM service (June 2026). The Brossard branch has been operationally stable since 2023, so we treat it as representative of pre-closure service levels.*

In [14]:
# load REM GTFS tables directly
REM_GTFS = Path("data/rem/gtfs")
rem_stops     = pd.read_csv(REM_GTFS / "stops.txt")
rem_trips     = pd.read_csv(REM_GTFS / "trips.txt")
rem_stop_times = pd.read_csv(REM_GTFS / "stop_times.txt")
rem_calendar  = pd.read_csv(REM_GTFS / "calendar.txt")

# Brossard branch stations, south to north
BRANCH_CODES = ["RIV", "DUQ", "PAN", "IDS", "GCT"]
BRANCH_NAMES = {
    "RIV": "Brossard",
    "DUQ": "Du Quartier",
    "PAN": "Panama",
    "IDS": "Île-des-Soeurs",
    "GCT": "Gare Centrale",
}

# platform-level stops on the branch (QUAI stops only, no entrances or turnaround zones)
branch_stops = rem_stops[
    rem_stops.stop_id.str.contains("QUAI") &
    rem_stops.stop_id.str.extract(r"_([A-Z]{3})$")[0].isin(BRANCH_CODES)
].copy()
branch_stops["station_code"] = branch_stops.stop_id.str.extract(r"_([A-Z]{3})$")

print(f"{len(branch_stops)} branch platform stops across {branch_stops.station_code.nunique()} stations")

11 branch platform stops across 5 stations


In [ ]:
# GTFS times can exceed 24:00:00 for post-midnight service
def to_minutes(t: str) -> float:
    h, m, s = map(int, t.split(":"))
    return h * 60 + m + s / 60

# filter stop_times to branch platforms and parse times
branch_st = rem_stop_times[rem_stop_times.stop_id.isin(branch_stops.stop_id)].copy()
branch_st["station_code"] = branch_st.stop_id.str.extract(r"_([A-Z]{3})$")
branch_st["dep_min"] = branch_st.departure_time.apply(to_minutes)
branch_st["arr_min"] = branch_st.arrival_time.apply(to_minutes)

# candidates: trips serving both Brossard and Gare Centrale
trips_at_riv = set(branch_st[branch_st.station_code == "RIV"].trip_id)
trips_at_gct = set(branch_st[branch_st.station_code == "GCT"].trip_id)
candidates = branch_st[branch_st.trip_id.isin(trips_at_riv & trips_at_gct)]

# northbound trips: Brossard stop_sequence < Gare Centrale stop_sequence
riv_seq = candidates[candidates.station_code == "RIV"].groupby("trip_id")["stop_sequence"].min()
gct_seq = candidates[candidates.station_code == "GCT"].groupby("trip_id")["stop_sequence"].min()
northbound_ids = riv_seq[riv_seq < gct_seq].index

print(f"Northbound trips identified: {len(northbound_ids)}")

nb_st = branch_st[branch_st.trip_id.isin(northbound_ids)].copy()

# elapsed time from Brossard departure; origin stop is 0 by definition
riv_dep = nb_st[nb_st.station_code == "RIV"].groupby("trip_id")["dep_min"].min()
nb_st["elapsed"] = nb_st["arr_min"] - nb_st.trip_id.map(riv_dep)
nb_st.loc[nb_st.station_code == "RIV", "elapsed"] = 0.0

# median elapsed time at each station across all northbound trips
profile = (
    nb_st
    .groupby("station_code")["elapsed"]
    .median()
    .reindex(BRANCH_CODES)
    .reset_index()
    .rename(columns={"elapsed": "min_from_brossard"})
)
profile["station"] = profile.station_code.map(BRANCH_NAMES)
profile["min_from_brossard"] = profile.min_from_brossard.round(1)

print()
print(profile[["station", "min_from_brossard"]].to_string(index=False))

Northbound trips identified: 536

       station  min_from_brossard
      Brossard                0.0
   Du Quartier                2.0
        Panama                5.7
Île-des-Soeurs               10.6
 Gare Centrale               17.8


In [19]:
# weekday service IDs from calendar
weekday_service_ids = rem_calendar[rem_calendar.monday == 1].service_id
weekday_trip_ids = rem_trips[rem_trips.service_id.isin(weekday_service_ids)].trip_id

# northbound weekday departures from Brossard, sorted chronologically
brossard_deps = (
    nb_st[
        nb_st.trip_id.isin(weekday_trip_ids) &
        (nb_st.station_code == "RIV")
    ]
    .drop_duplicates(subset=["trip_id"])
    .sort_values("dep_min")
    .copy()
)
brossard_deps["headway_min"] = brossard_deps.dep_min.diff()

# all-day (5am–11pm) and morning peak (7–9am)
allday = brossard_deps[brossard_deps.dep_min.between(5 * 60, 23 * 60)]
peak   = brossard_deps[brossard_deps.dep_min.between(7 * 60,  9 * 60)]

print("Weekday headways at Brossard (northbound):")
print(f"  All-day median:        {allday.headway_min.median():.1f} min")
print(f"  Morning peak median:   {peak.headway_min.median():.1f} min")
print(f"  Morning peak min:      {peak.headway_min.min():.1f} min")

Weekday headways at Brossard (northbound):
  All-day median:        3.5 min
  Morning peak median:   3.5 min
  Morning peak min:      3.5 min
